In [0]:
df = spark.readStream.table(
    "github_stream_event_catalog.bronze_layer.github_events_raw"
)


In [0]:
# df = spark.read.table('github_stream_event_catalog.bronze_layer.github_events_raw')
# display(df)

In [0]:
df.printSchema()

root
 |-- actor: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- display_login: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- org: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- before: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- full_ref: string (nullable = true)
 |    |-- head: string (nullable = true)
 |    |-- master_branch: string (nullable = true)
 |    |-- push_id: long (nullable = true)
 |    |-- pusher_type: string (nullable = true)
 |    |-- ref

In [0]:
# %sql
# SHOW TABLES IN github_stream_event_catalog.bronze_layer;

In [0]:
# df.count()

In [0]:
from pyspark.sql.functions import *

df_br = df.select(
    col('id').alias('event_id'),
    col('type'),
    col('created_at'),
    col('public'),

    col('actor.id').alias('actor_id'),
    col('actor.login').alias('actor_login'),
    col('actor.display_login').alias('actor_display_login'),
    
    col('repo.id').alias('repo_id'),
    col('repo.name').alias('repo_name'),
    # col('repo.url').alias('repo_url'),

    col('org.id').alias('org_id'),
    col('org.login').alias('org_login'),
    
    col('payload.push_id').alias('payload_push_id'),
    col('payload.ref').alias('payload_ref'),
    col('payload.ref_type').alias('payload_ref_type'),
    # col('payload.head').alias('payload_head'),
    # col('payload.before').alias('payload_before')
)

In [0]:
# display(df_br)

In [0]:
# df_br.count()


In [0]:
# df_br.groupby('event_id').count().filter(col('count') > 1).show(5)
# df_br.filter(col('event_id') == '9185255830').show()

In [0]:
silver_df = df_br.drop_duplicates(["event_id"])
# silver_df.groupby('event_id').count().filter(col('count') > 1).show(5)
# silver_df.count()

In [0]:
# null_counts = silver_df.select([
#     sum(col(c).isNull().cast("int")).alias(c) for c in silver_df.columns
# ])

# display(null_counts)

In [0]:
silver_df = silver_df.fillna({
    "org_login": "personal_repo",
    'org_id':0,
})
silver_df = silver_df.fillna({
    "payload_ref_type": "branch"
})
# null_count_fill = silver_df.select([
#     sum(col(c).isNull().cast("int")).alias(c) for c in silver_df.columns
# ])

# display(null_count_fill)

In [0]:
silver_df = silver_df.withColumn(
    "created_at",
    to_timestamp("created_at")
)

In [0]:
silver_df = silver_df.withColumn(
    "branch",
    regexp_replace("payload_ref", "refs/heads/", "")
)

In [0]:
# silver_df.select([
#     countDistinct(c).alias(c) for c in silver_df.columns
# ]).show()

In [0]:
# silver_df.groupBy("type").count().show()

In [0]:
# display(silver_df)

In [0]:
# existing_ids = spark.table(
#     "github_stream_event_catalog.silver_layer.silver_github_events"
# ).select("event_id")

# silver_df = silver_df.join(existing_ids, "event_id", "left_anti")

In [0]:
(silver_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/git_data_raw/bronze_layer_schema/git_data_raw/checkpoints/silver"
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .table("github_stream_event_catalog.silver_layer.silver_github_events")
)

In [0]:
# %sql
# create schema github_stream_event_catalog.silver_layer;

In [0]:
# silver_df.write.format('delta').mode("append").saveAsTable("github_stream_event_catalog.silver_layer.silver_github_events")